# VF-NeRF (conditional-NF fork) — Kaggle training

Trains (or restores) a **frozen nerfacto backbone** on the `bonsai` scene (Mip-NeRF
360), then trains the **conditional Normalizing Flow** that models
`P((3D point, 3D direction) | DINO feature)` against that frozen NeRF, and lets you
**probe it**: click up to 5 points on training images in cell 6a, then generate the
sampled novel views for them in cell 6b. Cell 4 (a spiral flythrough render) is
optional and auto-skips when nerfacto is restored from a checkpoint.

### Before you run — REQUIRED (open the right sidebar → Settings)
1. **Accelerator → GPU T4 x2** (or **GPU P100**). Cell 0 hard-stops if this is off.
2. **Internet → On** — needs a phone-verified Kaggle account
   (Settings → Phone Verification). Cell 0 hard-stops if this is off.
3. Then **Save Version → Save & Run All (Commit)** for an unattended run — Kaggle
   allows up to 12 h and, unlike free Colab, will not reclaim the GPU mid-run.
   Everything written to `/kaggle/working/` is saved as the version's Output.

Heavy build artifacts (venv, repo, dataset) go in `/kaggle/temp/` (not persisted).
Checkpoints (+ any renders) land in `/kaggle/working/` as they are produced.

Runtime: ~30–60 min conditional NF if you attach both checkpoints; add ~1 h if
nerfacto trains from scratch, plus ~10 min if you force the flythrough render.

In [ ]:
# @title 0. Environment sanity check  (STOP if this cell raises)
import sys, platform, subprocess, os
print('system python:', sys.version)
print('platform:', platform.platform())
smi = subprocess.run(['bash','-c','nvidia-smi -L 2>/dev/null || true'], capture_output=True, text=True).stdout.strip()
print('GPU:', smi or '(none)')
if not smi:
    raise RuntimeError(
        'No GPU. Right sidebar -> Settings -> Accelerator -> GPU T4 x2 (or P100), '
        'and Internet -> On (needs a phone-verified account). Then re-run.')
net = subprocess.run(['bash','-c','curl -sI --max-time 10 https://pypi.org >/dev/null && echo ok || echo fail'], capture_output=True, text=True).stdout.strip()
print('internet:', net)
if net != 'ok':
    raise RuntimeError('No internet. Right sidebar -> Settings -> Internet -> On, then re-run.')
for d in ('/kaggle/temp', '/kaggle/working'):
    os.makedirs(d, exist_ok=True)

In [ ]:
# @title 1. Build isolated CUDA 11.7 / Python 3.10 env + tiny-cuda-nn + repo
# Mirrors the debugged Colab setup: torch==1.13.1+cu117 has no cp311 wheel and
# tiny-cuda-nn needs the CUDA 11.7 dev headers, so we build a py3.10 venv rather
# than touch Kaggle's system interpreter.
setup = r'''#!/bin/bash
set -e
export DEBIAN_FRONTEND=noninteractive

TMP=/kaggle/temp
REPO=$TMP/VF-NeRF-conditional
VENV=$TMP/venv310
mkdir -p $TMP

echo '=== add NVIDIA CUDA apt repo (for the 11.7 dev packages) ==='
UBU=$(. /etc/os-release && echo ${VERSION_ID//./})   # 2204 / 2404
if ! ls /etc/apt/sources.list.d/ | grep -qi cuda; then
  wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu${UBU}/x86_64/cuda-keyring_1.1-1_all.deb || \
  wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
  dpkg -i /tmp/cuda-keyring.deb
fi

echo '=== apt packages ==='
apt-get -qq update
# python3.10: default on 22.04; via deadsnakes otherwise
if ! apt-get -qq install -y python3.10 python3.10-venv python3.10-dev 2>/dev/null; then
  apt-get -qq install -y software-properties-common
  add-apt-repository -y ppa:deadsnakes/ppa
  apt-get -qq update
  apt-get -qq install -y python3.10 python3.10-venv python3.10-dev
fi
apt-get -q install -y \
    cuda-nvcc-11-7 cuda-cudart-dev-11-7 cuda-nvrtc-dev-11-7 libcublas-dev-11-7 libcufft-dev-11-7 \
    libcurand-dev-11-7 libcusolver-dev-11-7 libcusparse-dev-11-7 libnpp-dev-11-7 \
    libnvjpeg-dev-11-7 ninja-build ffmpeg

if [ ! -x /usr/local/cuda-11.7/bin/nvcc ]; then
  echo 'FATAL: /usr/local/cuda-11.7/bin/nvcc missing after apt install'
  ls -R /usr/local/cuda-11.7 2>/dev/null | head -40; apt-cache policy cuda-nvcc-11-7
  exit 1
fi

echo '=== register CUDA 11.7 lib path ==='
echo '/usr/local/cuda-11.7/lib64' > /etc/ld.so.conf.d/cuda-11-7.conf && ldconfig
# tiny-cuda-nn links against the CUDA driver lib (-lcuda). apt CUDA has only the
# stub; the real one ships with the GPU driver as libcuda.so.1 with no .so symlink.
STUB=/usr/local/cuda-11.7/lib64/stubs
REAL=$(ldconfig -p | awk '/libcuda\.so\.1/{print $NF; exit}')
[ -n "$REAL" ] && [ ! -e /usr/local/cuda-11.7/lib64/libcuda.so ] && ln -sf "$REAL" /usr/local/cuda-11.7/lib64/libcuda.so
export LIBRARY_PATH=/usr/local/cuda-11.7/lib64:$STUB${LIBRARY_PATH:+:$LIBRARY_PATH}
echo "libcuda: real=$REAL  stub=$(ls $STUB/libcuda.so 2>/dev/null)"

echo '=== venv310 ==='
[ -d $VENV ] || python3.10 -m venv $VENV
$VENV/bin/pip install -q 'setuptools<81' wheel

echo '=== torch 1.13.1+cu117 ==='
$VENV/bin/pip install -q torch==1.13.1 torchvision functorch --extra-index-url https://download.pytorch.org/whl/cu117
$VENV/bin/pip install -q ninja

echo '=== toolchain diagnostics ==='
export CUDA_HOME=/usr/local/cuda-11.7
export PATH=/usr/local/cuda-11.7/bin:$PATH
ls -d /usr/local/cuda* || true
which nvcc; nvcc --version 2>&1 | tail -2 || echo 'NO nvcc at /usr/local/cuda-11.7/bin'
gcc --version | head -1; g++ --version | head -1
# CUDA 11.7 nvcc rejects gcc>11; pin to gcc-11 if a newer default is present
if gcc -dumpversion | grep -qvE '^(9|10|11)'; then
  apt-get -qq install -y gcc-11 g++-11
  export CC=gcc-11 CXX=g++-11
  export NVCC_PREPEND_FLAGS='-ccbin g++-11'
  echo 'pinned to gcc-11'
fi

echo '=== build tiny-cuda-nn ==='
# derive the arch from the actual GPU (T4 -> 75, P100 -> 60, L4 -> 89, ...)
GPUCC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '. ')
export TCNN_CUDA_ARCHITECTURES=${GPUCC:-75}
echo "TCNN_CUDA_ARCHITECTURES=$TCNN_CUDA_ARCHITECTURES"
# master built fine on Colab recently; fall back through the last few tags if it
# has since moved past CUDA 11.7. Override the whole list with TCNN_REFS.
TCNN_OK=
for ref in ${TCNN_REFS:-master v1.6 v1.5}; do
  [ "$ref" = master ] && spec='git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch' \
                       || spec="git+https://github.com/NVlabs/tiny-cuda-nn/@${ref}#subdirectory=bindings/torch"
  echo "--- trying tiny-cuda-nn @ $ref ---" | tee -a /kaggle/working/tcnn_build.log
  if $VENV/bin/pip install --no-build-isolation -v "$spec" >> /kaggle/working/tcnn_build.log 2>&1; then
    TCNN_OK=$ref; break
  fi
  echo "  @ $ref failed:"; grep -E 'error:|fatal error|unsupported|cannot find -l|undefined reference|ld returned' /kaggle/working/tcnn_build.log | tail -8
done
if [ -z "$TCNN_OK" ]; then
  echo '!!! tiny-cuda-nn build FAILED for every ref — full log at /kaggle/working/tcnn_build.log'
  grep -nE 'error:|fatal error|unsupported|cannot find -l|undefined reference|ld returned' /kaggle/working/tcnn_build.log | tail -40
  exit 1
fi
echo "tiny-cuda-nn OK (@ $TCNN_OK)"
$VENV/bin/python -c 'import tinycudann as t; print("tcnn import ok", t.__version__ if hasattr(t,"__version__") else "")'

echo '=== clone repo ==='
rm -rf $REPO
git clone --quiet https://github.com/itayhanoch/VF-NeRF-conditional.git $REPO

echo '=== apply known repo fixes ==='
# 1) eval call site missing the `step` arg (crashes at the first full-image eval)
sed -i 's/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch)/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch, step)/' \
    $REPO/nerfstudio/pipelines/base_pipeline.py
# 2) scripts/render.py imports get_mask_from_view_likelihood, removed with the
#    registration pipeline — dead import, never used. Strip it so ns-render works.
sed -i '/get_mask_from_view_likelihood/d' $REPO/scripts/render.py
# 3) DINOv2 hub code (facebookresearch/dinov2 @ main) now needs torch>=2.0
#    (F.scaled_dot_product_attention). Prepend a math-identical fallback to the
#    one module that torch.hub.load's DINOv2 — nerfstudio/utils/dino_features.py.
$VENV/bin/python - "$REPO/nerfstudio/utils/dino_features.py" <<'PYEOF'
import sys, pathlib
p = pathlib.Path(sys.argv[1])
s = p.read_text()
if 'sdpa-shim' not in s:
    shim = ('\n# sdpa-shim (DINOv2 main needs torch>=2.0; this stack is torch 1.13)\n'
            'import math as _m\n'
            'import torch as _t\n'
            'import torch.nn.functional as _F\n'
            'if not hasattr(_F, "scaled_dot_product_attention"):\n'
            '    def _sdpa(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):\n'
            '        sc = 1.0 / _m.sqrt(q.size(-1)) if scale is None else scale\n'
            '        a = _t.matmul(q, k.transpose(-2, -1)) * sc\n'
            '        if attn_mask is not None:\n'
            '            a = a.masked_fill(~attn_mask, float("-inf")) if attn_mask.dtype == _t.bool else a + attn_mask\n'
            '        a = a.softmax(-1)\n'
            '        if dropout_p:\n'
            '            a = _F.dropout(a, dropout_p)\n'
            '        return _t.matmul(a, v)\n'
            '    _F.scaled_dot_product_attention = _sdpa\n')
    # insert AFTER `from __future__` (must stay first statement); else after the docstring
    anchor = 'from __future__ import annotations\n'
    if anchor in s:
        s = s.replace(anchor, anchor + shim, 1)
    else:
        s = shim.lstrip() + '\n' + s
    p.write_text(s)
    print('dino_features.py: sdpa shim installed')
else:
    print('dino_features.py: sdpa shim already present')
PYEOF
# 4) conditional-NF trainer: cap DINO res, store features at PATCH resolution
#    (a per-pixel map per image is hundreds of GB), park the frozen NeRF on CPU
#    during precompute, and build cameras in the checkpoint's own dataparser frame.
#    (patch_dino_oom.py is written from base64 by the Python wrapper below --
#    it has triple-quoted strings so it cannot live inside this bash heredoc.)
$VENV/bin/python /kaggle/temp/patch_dino_oom.py "$REPO"

echo '=== install repo + normalizing-flows ==='
cd $REPO
$VENV/bin/pip install -q --no-build-isolation -e . -e ./normalizing-flows

echo '=== download bonsai scene (~1GB via HTTP range requests) ==='
[ -d data/mipnerf360/bonsai/images ] || \
    $VENV/bin/python scripts/downloads/download_mipnerf360.py --scene bonsai --save-dir data/mipnerf360

echo '=== generate downscaled image folders (x2 for training, x4 spare) ==='
# nerfstudio 0.2.1 does NOT auto-generate these; the dataparser just looks for
# data_dir/images_<N>/. Original VF-NeRF trained at downscale 2; full res OOMs
# the image cache on a free-tier box.
$VENV/bin/python - <<'PYEOF'
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
src = Path('data/mipnerf360/bonsai/images')
n_src = len(list(src.glob('*')))
for factor in (2, 4):
    dst = Path(f'data/mipnerf360/bonsai/images_{factor}')
    if dst.is_dir() and len(list(dst.glob('*'))) == n_src:
        print(f'images_{factor} present, skip'); continue
    dst.mkdir(exist_ok=True)
    def f(p, factor=factor, dst=dst):
        im = Image.open(p); w, h = im.size
        im.resize((w // factor, h // factor), Image.LANCZOS).save(dst / p.name)
    with ThreadPoolExecutor(max_workers=8) as ex:
        list(ex.map(f, sorted(src.glob('*'))))
    print(f'images_{factor}:', len(list(dst.glob('*'))))
PYEOF
echo '=== SETUP COMPLETE ==='
'''
import os, subprocess, base64
os.makedirs('/kaggle/temp', exist_ok=True)
# patch_dino_oom.py has ''' triple-quotes -> ship it as base64, not inline
_PATCH_B64 = 'IiIiUGF0Y2ggdGhlIGNvbmRpdGlvbmFsLU5GIHRyYWluZXIgZm9yIGEgMTYgR0IgLyB0b3JjaC0xLjEzIGJveC4gIFJ1bjogcHl0aG9uIHBhdGNoX2Rpbm9fb29tLnB5IDxyZXBvX3Jvb3Q+CgpSb290IGNhdXNlIG9mIHRoZSBDVURBIE9PTXMgaW4gc2NyaXB0cy90cmFpbl9jb25kaXRpb25hbF9uZi5weToKICAqIERJTk92MiByYW4gYXQgbmVhci1uYXRpdmUgcmVzIGFuZCB0aGUgdG9yY2gtMS4xMyBhdHRlbnRpb24gZmFsbGJhY2sgYnVpbGRzIHRoZQogICAgZnVsbCBwYXRjaCB4IHBhdGNoIG1hdHJpeCAofjEuNiBHQikuCiAgKiBwcmVjb21wdXRlX2Rpbm9fY2FjaGUga2VwdCBhIFBFUi1QSVhFTCBmZWF0dXJlIG1hcCBmb3IgRVZFUlkgdHJhaW5pbmcgaW1hZ2UKICAgIHJlc2lkZW50IGF0IG9uY2UgLT4gMjYzIHggMzg0IHggMTAzOCB4IDE1NTQgeCA0IEIgfj0gNjUwIEdCLgogICogdGhlIGZyb3plbiBOZVJGICh+MTIgR0IpIHN0YXllZCBvbiB0aGUgR1BVIHRocm91Z2hvdXQgdGhlIERJTk8gcHJlY29tcHV0ZS4KICAqIGJ1aWxkX3RyYWluaW5nX2NhbWVyYXMgdXNlZCBzdG9jayBkYXRhcGFyc2VyIGRlZmF1bHRzLCBub3QgdGhlIGNoZWNrcG9pbnQncwogICAgY2VudGVyLW1ldGhvZCAvIHNjZW5lLXNjYWxlIC0+IGNhbWVyYXMgaW4gdGhlIHdyb25nIGZyYW1lLgoKRml4ZXM6CiAgMS4gZGlub19mZWF0dXJlcy5weTogY2FwIHRoZSBESU5PIGZvcndhcmQgcGFzcyBhdCBNQVhfRElOT19TSURFOyBpbnRlcnBvbGF0ZSBvbiBDUFUuCiAgMi4gdHJhaW5fY29uZGl0aW9uYWxfbmYucHk6IHN0b3JlIERJTk8gZmVhdHVyZXMgYXQgUEFUQ0ggcmVzb2x1dGlvbiAofjkyeDYxKSwgZnAxNiwKICAgICBvbiBDUFUgKH4xIEdCIHRvdGFsKTsgc2FtcGxlX2JhdGNoIG1hcHMgYSBzYW1wbGVkIHBpeGVsIHRvIGl0cyBwYXRjaCBjZWxsLgogIDMuIHRyYWluX2NvbmRpdGlvbmFsX25mLnB5OiBwYXJrIHRoZSBOZVJGIG9uIENQVSBkdXJpbmcgdGhlIHByZWNvbXB1dGU7IHJlYnVpbGQgaXQKICAgICBvbiBHUFUgZm9yIHRoZSB0cmFpbmluZyBsb29wLgogIDQuIHRyYWluX2NvbmRpdGlvbmFsX25mLnB5OiBidWlsZCBjYW1lcmFzIGZyb20gdGhlIGNoZWNrcG9pbnQncyBvd24gZGF0YXBhcnNlciBjb25maWcuCiIiIgppbXBvcnQgc3lzLCBwYXRobGliLCBhc3QKCnJlcG8gPSBwYXRobGliLlBhdGgoc3lzLmFyZ3ZbMV0pCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gZGlub19mZWF0dXJlcy5weQpwID0gcmVwbyAvICJuZXJmc3R1ZGlvL3V0aWxzL2Rpbm9fZmVhdHVyZXMucHkiCnMgPSBwLnJlYWRfdGV4dCgpCgppZiAiTUFYX0RJTk9fU0lERSIgbm90IGluIHM6CiAgICBzID0gcy5yZXBsYWNlKAogICAgICAgICJFTUJFRF9ESU0gPSAzODRcbiIsCiAgICAgICAgIkVNQkVEX0RJTSA9IDM4NFxuTUFYX0RJTk9fU0lERSA9IDEyODggICMgY2FwIERJTk8gZm9yd2FyZC1wYXNzIHJlcyAobm8gZmxhc2gtYXR0biBvbiB0b3JjaCAxLjEzKVxuIiwKICAgICAgICAxLAogICAgKQogICAgb2xkX2VwZyA9ICgKICAgICAgICAiICAgICAgICBwYWRkZWQsIChoLCB3KSA9IF9wYWRfdG9fcGF0Y2hfbXVsdGlwbGUoaW1hZ2VfY2h3XzAxLnRvKHNlbGYuZGV2aWNlKSlcbiIKICAgICAgICAiICAgICAgICBub3JtYWxpemVkID0gX25vcm1hbGl6ZV9mb3JfZGlub3YyKHBhZGRlZClcbiIKICAgICAgICAiICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKTpcbiIKICAgICAgICAiICAgICAgICAgICAgKHBhdGNoX3Rva2VucywpID0gc2VsZi5tb2RlbC5nZXRfaW50ZXJtZWRpYXRlX2xheWVycyhub3JtYWxpemVkLnVuc3F1ZWV6ZSgwKSwgbj0xLCByZXNoYXBlPVRydWUsIHJldHVybl9jbGFzc190b2tlbj1GYWxzZSlcbiIKICAgICAgICAiICAgICAgICByZXR1cm4gcGF0Y2hfdG9rZW5zLnNxdWVlemUoMCksIChoLCB3KSAgIyBbRU1CRURfRElNLCBIcCwgV3BdLCAoaCwgdylcbiIKICAgICkKICAgIG5ld19lcGcgPSAoCiAgICAgICAgIiAgICAgICAgaW1nID0gaW1hZ2VfY2h3XzAxLnRvKHNlbGYuZGV2aWNlKVxuIgogICAgICAgICIgICAgICAgIG9yaWdfaCwgb3JpZ193ID0gaW50KGltZy5zaGFwZVstMl0pLCBpbnQoaW1nLnNoYXBlWy0xXSlcbiIKICAgICAgICAiICAgICAgICBsb25nZXN0ID0gbWF4KG9yaWdfaCwgb3JpZ193KVxuIgogICAgICAgICIgICAgICAgIGlmIGxvbmdlc3QgPiBNQVhfRElOT19TSURFOlxuIgogICAgICAgICIgICAgICAgICAgICBzYyA9IE1BWF9ESU5PX1NJREUgLyBsb25nZXN0XG4iCiAgICAgICAgIiAgICAgICAgICAgIGltZyA9IEYuaW50ZXJwb2xhdGUoaW1nLnVuc3F1ZWV6ZSgwKSwgc2NhbGVfZmFjdG9yPXNjLCBtb2RlPVwiYmlsaW5lYXJcIixcbiIKICAgICAgICAiICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhbGlnbl9jb3JuZXJzPUZhbHNlLCByZWNvbXB1dGVfc2NhbGVfZmFjdG9yPUZhbHNlKS5zcXVlZXplKDApXG4iCiAgICAgICAgIiAgICAgICAgcGFkZGVkLCBfID0gX3BhZF90b19wYXRjaF9tdWx0aXBsZShpbWcpXG4iCiAgICAgICAgIiAgICAgICAgbm9ybWFsaXplZCA9IF9ub3JtYWxpemVfZm9yX2Rpbm92MihwYWRkZWQpXG4iCiAgICAgICAgIiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6XG4iCiAgICAgICAgIiAgICAgICAgICAgIChwYXRjaF90b2tlbnMsKSA9IHNlbGYubW9kZWwuZ2V0X2ludGVybWVkaWF0ZV9sYXllcnMobm9ybWFsaXplZC51bnNxdWVlemUoMCksIG49MSwgcmVzaGFwZT1UcnVlLCByZXR1cm5fY2xhc3NfdG9rZW49RmFsc2UpXG4iCiAgICAgICAgIiAgICAgICAgcmV0dXJuIHBhdGNoX3Rva2Vucy5zcXVlZXplKDApLCAob3JpZ19oLCBvcmlnX3cpXG4iCiAgICApCiAgICBhc3NlcnQgb2xkX2VwZyBpbiBzCiAgICBzID0gcy5yZXBsYWNlKG9sZF9lcGcsIG5ld19lcGcsIDEpCiAgICBvbGRfdXBzID0gKAogICAgICAgICIgICAgICAgIGhwLCB3cCA9IHBhdGNoX2dyaWQuc2hhcGVbLTI6XVxuIgogICAgICAgICIgICAgICAgIHBhZGRlZF9oLCBwYWRkZWRfdyA9IGhwICogUEFUQ0hfU0laRSwgd3AgKiBQQVRDSF9TSVpFXG4iCiAgICAgICAgIiAgICAgICAgdXBzYW1wbGVkID0gRi5pbnRlcnBvbGF0ZShwYXRjaF9ncmlkLnVuc3F1ZWV6ZSgwKSwgc2l6ZT0ocGFkZGVkX2gsIHBhZGRlZF93KSwgbW9kZT1cImJpbGluZWFyXCIsIGFsaWduX2Nvcm5lcnM9RmFsc2UpLnNxdWVlemUoMClcbiIKICAgICAgICAiICAgICAgICByZXR1cm4gdXBzYW1wbGVkWzosIDpoLCA6d11cbiIKICAgICkKICAgIG5ld191cHMgPSAoCiAgICAgICAgIiAgICAgICAgZyA9IHBhdGNoX2dyaWQuZGV0YWNoKCkuZmxvYXQoKS5jcHUoKS51bnNxdWVlemUoMClcbiIKICAgICAgICAiICAgICAgICByZXR1cm4gRi5pbnRlcnBvbGF0ZShnLCBzaXplPShpbnQoaCksIGludCh3KSksIG1vZGU9XCJiaWxpbmVhclwiLCBhbGlnbl9jb3JuZXJzPUZhbHNlKS5zcXVlZXplKDApXG4iCiAgICApCiAgICBhc3NlcnQgb2xkX3VwcyBpbiBzCiAgICBzID0gcy5yZXBsYWNlKG9sZF91cHMsIG5ld191cHMsIDEpCiAgICBwLndyaXRlX3RleHQocykKICAgIHByaW50KCJkaW5vX2ZlYXR1cmVzLnB5OiByZXNvbHV0aW9uIGNhcCArIENQVSB1cHNhbXBsZSIpCgphc3QucGFyc2UocC5yZWFkX3RleHQoKSkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSB0cmFpbl9jb25kaXRpb25hbF9uZi5weQpwID0gcmVwbyAvICJzY3JpcHRzL3RyYWluX2NvbmRpdGlvbmFsX25mLnB5IgpzID0gcC5yZWFkX3RleHQoKQoKb2xkX2Jsb2NrID0gKAogICAgImRlZiBidWlsZF90cmFpbmluZ19jYW1lcmFzKHNjZW5lX2RpcjogUGF0aCk6XG4iCiAgICAiICAgIGRhdGFwYXJzZXIgPSBOZXJmc3R1ZGlvRGF0YVBhcnNlckNvbmZpZyhkYXRhPXNjZW5lX2Rpcikuc2V0dXAoKVxuIgogICAgIiAgICBvdXRwdXRzID0gZGF0YXBhcnNlci5nZXRfZGF0YXBhcnNlcl9vdXRwdXRzKHNwbGl0PVwidHJhaW5cIilcbiIKICAgICIgICAgcmV0dXJuIG91dHB1dHMuY2FtZXJhcywgb3V0cHV0cy5pbWFnZV9maWxlbmFtZXNcbiIKICAgICJcbiIKICAgICJcbiIKICAgICJkZWYgcHJlY29tcHV0ZV9kaW5vX2NhY2hlKGltYWdlX2ZpbGVuYW1lcywgY2FjaGVfZGlyOiBQYXRoLCBleHRyYWN0b3I6IERpbm9FeHRyYWN0b3IpOlxuIgogICAgIiAgICBcIlwiXCJSZXR1cm5zIGEgbGlzdCBvZiBbRU1CRURfRElNLEgsV10gZmVhdHVyZS1tYXAgdGVuc29ycywgYWxpZ25lZCB3aXRoIGltYWdlX2ZpbGVuYW1lcy5cIlwiXCJcbiIKICAgICIgICAgY2FjaGVzID0gW11cbiIKICAgICIgICAgZm9yIGksIHBhdGggaW4gZW51bWVyYXRlKGltYWdlX2ZpbGVuYW1lcyk6XG4iCiAgICAiICAgICAgICBpbWcgPSBsb2FkX2ltYWdlX2Nod18wMShwYXRoKVxuIgogICAgIiAgICAgICAgZmVhdCA9IGdldF9vcl9jb21wdXRlX2NhY2hlKGltZywgcGF0aC5zdGVtLCBjYWNoZV9kaXIsIGV4dHJhY3RvcilcbiIKICAgICIgICAgICAgIGNhY2hlcy5hcHBlbmQoZmVhdClcbiIKICAgICIgICAgICAgIGlmIChpICsgMSkgJSAyMCA9PSAwIG9yIChpICsgMSkgPT0gbGVuKGltYWdlX2ZpbGVuYW1lcyk6XG4iCiAgICAiICAgICAgICAgICAgcHJpbnQoZlwiICBESU5PIGNhY2hlOiB7aSArIDF9L3tsZW4oaW1hZ2VfZmlsZW5hbWVzKX1cIilcbiIKICAgICIgICAgcmV0dXJuIGNhY2hlc1xuIgogICAgIlxuIgogICAgIlxuIgogICAgImRlZiBzYW1wbGVfYmF0Y2goY2FtZXJhcywgZGlub19jYWNoZXMsIGJhdGNoX3NpemUsIGRldmljZSk6XG4iCiAgICAiICAgIFwiXCJcIlJhbmRvbSAoaW1hZ2UsIHBpeGVsKSB0cmlwbGVzIC0+IChSYXlCdW5kbGUsIGNvbmRpdGlvbltCLENdKSBvbiBgZGV2aWNlYC5cIlwiXCJcbiIKICAgICIgICAgbnVtX2ltYWdlcyA9IGxlbihkaW5vX2NhY2hlcylcbiIKICAgICIgICAgaW1nX2lkeCA9IHRvcmNoLnJhbmRpbnQoMCwgbnVtX2ltYWdlcywgKGJhdGNoX3NpemUsKSlcbiIKICAgICIgICAgeXMgPSB0b3JjaC5lbXB0eShiYXRjaF9zaXplLCBkdHlwZT10b3JjaC5sb25nKVxuIgogICAgIiAgICB4cyA9IHRvcmNoLmVtcHR5KGJhdGNoX3NpemUsIGR0eXBlPXRvcmNoLmxvbmcpXG4iCiAgICAiICAgIGVtYmVkX2RpbSA9IGRpbm9fY2FjaGVzWzBdLnNoYXBlWzBdXG4iCiAgICAiICAgIGNvbmRpdGlvbnMgPSB0b3JjaC5lbXB0eShiYXRjaF9zaXplLCBlbWJlZF9kaW0pXG4iCiAgICAiXG4iCiAgICAiICAgIGZvciBpIGluIHJhbmdlKG51bV9pbWFnZXMpOlxuIgogICAgIiAgICAgICAgbWFzayA9IGltZ19pZHggPT0gaVxuIgogICAgIiAgICAgICAgbiA9IGludChtYXNrLnN1bSgpKVxuIgogICAgIiAgICAgICAgaWYgbiA9PSAwOlxuIgogICAgIiAgICAgICAgICAgIGNvbnRpbnVlXG4iCiAgICAiICAgICAgICBoLCB3ID0gZGlub19jYWNoZXNbaV0uc2hhcGVbLTI6XVxuIgogICAgIiAgICAgICAgeSA9IHRvcmNoLnJhbmRpbnQoMCwgaCwgKG4sKSlcbiIKICAgICIgICAgICAgIHggPSB0b3JjaC5yYW5kaW50KDAsIHcsIChuLCkpXG4iCiAgICAiICAgICAgICB5c1ttYXNrXSA9IHlcbiIKICAgICIgICAgICAgIHhzW21hc2tdID0geFxuIgogICAgIiAgICAgICAgY29uZGl0aW9uc1ttYXNrXSA9IGRpbm9fY2FjaGVzW2ldWzosIHksIHhdLnBlcm11dGUoMSwgMCkuY3B1KClcbiIKICAgICJcbiIKICAgICIgICAgY29vcmRzID0gdG9yY2guc3RhY2soW3lzLmZsb2F0KCkgKyAwLjUsIHhzLmZsb2F0KCkgKyAwLjVdLCBkaW09LTEpICAjICh5LCB4KSwgbWF0Y2hlcyBDYW1lcmFzLmdlbmVyYXRlX3JheXMnIGNvbnZlbnRpb25cbiIKICAgICIgICAgY2FtZXJhX2luZGljZXMgPSBpbWdfaWR4LnVuc3F1ZWV6ZSgtMSlcbiIKICAgICIgICAgcmF5X2J1bmRsZSA9IGNhbWVyYXMuZ2VuZXJhdGVfcmF5cyhjYW1lcmFfaW5kaWNlcz1jYW1lcmFfaW5kaWNlcywgY29vcmRzPWNvb3JkcylcbiIKICAgICIgICAgcmV0dXJuIHJheV9idW5kbGUudG8oZGV2aWNlKSwgY29uZGl0aW9ucy50byhkZXZpY2UpXG4iCikKCm5ld19ibG9jayA9ICcnJ2RlZiBidWlsZF90cmFpbmluZ19jYW1lcmFzKHNjZW5lX2RpcjogUGF0aCwgZGF0YXBhcnNlcl9jb25maWc9Tm9uZSk6CiAgICAjIG1hdGNoIHRoZSBmcm96ZW4gTmVSRidzIG93biBkYXRhcGFyc2VyIGZyYW1lIChjZW50ZXItbWV0aG9kIC8gc2NlbmUtc2NhbGUgLyBkb3duc2NhbGUpCiAgICBpZiBkYXRhcGFyc2VyX2NvbmZpZyBpcyBOb25lOgogICAgICAgIGRhdGFwYXJzZXJfY29uZmlnID0gTmVyZnN0dWRpb0RhdGFQYXJzZXJDb25maWcoKQogICAgZGF0YXBhcnNlcl9jb25maWcuZGF0YSA9IFBhdGgoc2NlbmVfZGlyKQogICAgb3V0cHV0cyA9IGRhdGFwYXJzZXJfY29uZmlnLnNldHVwKCkuZ2V0X2RhdGFwYXJzZXJfb3V0cHV0cyhzcGxpdD0idHJhaW4iKQogICAgcmV0dXJuIG91dHB1dHMuY2FtZXJhcywgb3V0cHV0cy5pbWFnZV9maWxlbmFtZXMKCgpkZWYgcHJlY29tcHV0ZV9kaW5vX2NhY2hlKGltYWdlX2ZpbGVuYW1lcywgY2FjaGVfZGlyOiBQYXRoLCBleHRyYWN0b3I6IERpbm9FeHRyYWN0b3IpOgogICAgIiIiUmV0dXJucyBbKHBhdGNoX2dyaWRbRU1CRURfRElNLEhwLFdwXSBmcDE2IENQVSwgKEgsIFcpKV0gcGVyIGltYWdlLgoKICAgIFN0b3JlcyBESU5PIGZlYXR1cmVzIGF0IFBBVENIIHJlc29sdXRpb24sIG5vdCBwZXItcGl4ZWwgLS0gYSBwZXItcGl4ZWwgbWFwIGZvcgogICAgZXZlcnkgdHJhaW5pbmcgaW1hZ2UgYXQgb25jZSBpcyBodW5kcmVkcyBvZiBHQi4gc2FtcGxlX2JhdGNoIG1hcHMgYSBzYW1wbGVkCiAgICBwaXhlbCB0byBpdHMgcGF0Y2ggY2VsbC4KICAgICIiIgogICAgZnJvbSBuZXJmc3R1ZGlvLnV0aWxzLmRpbm9fZmVhdHVyZXMgaW1wb3J0IGNhY2hlX3BhdGgKICAgIGNhY2hlcyA9IFtdCiAgICBmb3IgaSwgcGF0aCBpbiBlbnVtZXJhdGUoaW1hZ2VfZmlsZW5hbWVzKToKICAgICAgICBpbWcgPSBsb2FkX2ltYWdlX2Nod18wMShwYXRoKQogICAgICAgIGgsIHcgPSBpbnQoaW1nLnNoYXBlWy0yXSksIGludChpbWcuc2hhcGVbLTFdKQogICAgICAgIGNwID0gY2FjaGVfcGF0aChjYWNoZV9kaXIsIGV4dHJhY3Rvci5tb2RlbF9uYW1lLCBwYXRoLnN0ZW0pCiAgICAgICAgaWYgY3AuZXhpc3RzKCk6CiAgICAgICAgICAgIGdyaWQgPSB0b3JjaC5sb2FkKGNwLCBtYXBfbG9jYXRpb249ImNwdSIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgICAgICBncmlkLCBfID0gZXh0cmFjdG9yLmV4dHJhY3RfcGF0Y2hfZ3JpZChpbWcpCiAgICAgICAgICAgIGdyaWQgPSBncmlkLnRvKHRvcmNoLmZsb2F0MTYpLmNwdSgpCiAgICAgICAgICAgIGNwLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIHRvcmNoLnNhdmUoZ3JpZCwgY3ApCiAgICAgICAgY2FjaGVzLmFwcGVuZCgoZ3JpZCwgKGgsIHcpKSkKICAgICAgICBpZiAoaSArIDEpICUgMjAgPT0gMCBvciAoaSArIDEpID09IGxlbihpbWFnZV9maWxlbmFtZXMpOgogICAgICAgICAgICBwcmludChmIiAgRElOTyBjYWNoZToge2kgKyAxfS97bGVuKGltYWdlX2ZpbGVuYW1lcyl9IikKICAgIHJldHVybiBjYWNoZXMKCgpkZWYgc2FtcGxlX2JhdGNoKGNhbWVyYXMsIGRpbm9fY2FjaGVzLCBiYXRjaF9zaXplLCBkZXZpY2UpOgogICAgIiIiUmFuZG9tIChpbWFnZSwgcGF0Y2gpIHRyaXBsZXMgLT4gKFJheUJ1bmRsZSwgY29uZGl0aW9uW0IsQ10pIG9uIGBkZXZpY2VgLiIiIgogICAgbnVtX2ltYWdlcyA9IGxlbihkaW5vX2NhY2hlcykKICAgIGltZ19pZHggPSB0b3JjaC5yYW5kaW50KDAsIG51bV9pbWFnZXMsIChiYXRjaF9zaXplLCkpCiAgICB5cyA9IHRvcmNoLmVtcHR5KGJhdGNoX3NpemUpCiAgICB4cyA9IHRvcmNoLmVtcHR5KGJhdGNoX3NpemUpCiAgICBlbWJlZF9kaW0gPSBkaW5vX2NhY2hlc1swXVswXS5zaGFwZVswXQogICAgY29uZGl0aW9ucyA9IHRvcmNoLmVtcHR5KGJhdGNoX3NpemUsIGVtYmVkX2RpbSkKCiAgICBmb3IgaSBpbiByYW5nZShudW1faW1hZ2VzKToKICAgICAgICBtYXNrID0gaW1nX2lkeCA9PSBpCiAgICAgICAgbiA9IGludChtYXNrLnN1bSgpKQogICAgICAgIGlmIG4gPT0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBncmlkLCAoaCwgdykgPSBkaW5vX2NhY2hlc1tpXQogICAgICAgIGhwLCB3cCA9IGdyaWQuc2hhcGVbLTI6XQogICAgICAgIHB5ID0gdG9yY2gucmFuZGludCgwLCBocCwgKG4sKSkKICAgICAgICBweCA9IHRvcmNoLnJhbmRpbnQoMCwgd3AsIChuLCkpCiAgICAgICAgY29uZGl0aW9uc1ttYXNrXSA9IGdyaWRbOiwgcHksIHB4XS5wZXJtdXRlKDEsIDApLmZsb2F0KCkKICAgICAgICB5c1ttYXNrXSA9IChweS5mbG9hdCgpICsgMC41KSAqIChoIC8gaHApCiAgICAgICAgeHNbbWFza10gPSAocHguZmxvYXQoKSArIDAuNSkgKiAodyAvIHdwKQoKICAgIGNvb3JkcyA9IHRvcmNoLnN0YWNrKFt5cywgeHNdLCBkaW09LTEpICAjICh5LCB4KSBwaXhlbCBjb29yZHMgYXQgcGF0Y2gtY2VsbCBjZW50ZXJzCiAgICBjYW1lcmFfaW5kaWNlcyA9IGltZ19pZHgudW5zcXVlZXplKC0xKQogICAgcmF5X2J1bmRsZSA9IGNhbWVyYXMuZ2VuZXJhdGVfcmF5cyhjYW1lcmFfaW5kaWNlcz1jYW1lcmFfaW5kaWNlcywgY29vcmRzPWNvb3JkcykKICAgIHJldHVybiByYXlfYnVuZGxlLnRvKGRldmljZSksIGNvbmRpdGlvbnMudG8oZGV2aWNlKQonJycKCmFzc2VydCBvbGRfYmxvY2sgaW4gcywgImZ1bmN0aW9uIGJsb2NrIG5vdCBmb3VuZCAtLSBhbHJlYWR5IHBhdGNoZWQgb3IgZmlsZSBjaGFuZ2VkIgpzID0gcy5yZXBsYWNlKG9sZF9ibG9jaywgbmV3X2Jsb2NrLCAxKQoKcyA9IHMucmVwbGFjZSgKICAgICIgICAgXywgcGlwZWxpbmUsIF8sIF8gPSBldmFsX3NldHVwKGFyZ3MubmVyZl9jb25maWcsIHRlc3RfbW9kZT1cImluZmVyZW5jZVwiKVxuIiwKICAgICIgICAgY29uZmlnLCBwaXBlbGluZSwgXywgXyA9IGV2YWxfc2V0dXAoYXJncy5uZXJmX2NvbmZpZywgdGVzdF9tb2RlPVwiaW5mZXJlbmNlXCIpXG4iLAogICAgMSwKKQpzID0gcy5yZXBsYWNlKAogICAgIiAgICBmb3IgcGFyYW0gaW4gbmVyZl9tb2RlbC5wYXJhbWV0ZXJzKCk6XG4iCiAgICAiICAgICAgICBwYXJhbS5yZXF1aXJlc19ncmFkXyhGYWxzZSlcblxuIgogICAgIiAgICBwcmludChmXCJMb2FkaW5nIHRyYWluaW5nIGNhbWVyYXMvaW1hZ2VzIGZyb20ge2FyZ3Muc2NlbmVfZGlyfSAuLi5cIilcbiIKICAgICIgICAgY2FtZXJhcywgaW1hZ2VfZmlsZW5hbWVzID0gYnVpbGRfdHJhaW5pbmdfY2FtZXJhcyhhcmdzLnNjZW5lX2RpcilcbiIsCiAgICAiICAgIGZvciBwYXJhbSBpbiBuZXJmX21vZGVsLnBhcmFtZXRlcnMoKTpcbiIKICAgICIgICAgICAgIHBhcmFtLnJlcXVpcmVzX2dyYWRfKEZhbHNlKVxuXG4iCiAgICAiICAgICMgRElOTyBwcmVjb21wdXRlIHBlYWtzIEdQVSBtZW1vcnkgKG5vIGZsYXNoLWF0dG4gb24gdG9yY2ggMS4xMyk7IHBhcmsgdGhlXG4iCiAgICAiICAgICMgZnJvemVuIE5lUkYgb24gQ1BVIG1lYW53aGlsZSBhbmQgcmVidWlsZCBpdCBvbiBHUFUgZm9yIHRoZSB0cmFpbmluZyBsb29wLlxuIgogICAgIiAgICBuZXJmX21vZGVsID0gbmVyZl9tb2RlbC5jcHUoKVxuIgogICAgIiAgICB0b3JjaC5jdWRhLmVtcHR5X2NhY2hlKClcblxuIgogICAgIiAgICBwcmludChmXCJMb2FkaW5nIHRyYWluaW5nIGNhbWVyYXMvaW1hZ2VzIGZyb20ge2FyZ3Muc2NlbmVfZGlyfSAuLi5cIilcbiIKICAgICIgICAgY2FtZXJhcywgaW1hZ2VfZmlsZW5hbWVzID0gYnVpbGRfdHJhaW5pbmdfY2FtZXJhcyhhcmdzLnNjZW5lX2RpciwgY29uZmlnLnBpcGVsaW5lLmRhdGFtYW5hZ2VyLmRhdGFwYXJzZXIpXG4iLAogICAgMSwKKQpzID0gcy5yZXBsYWNlKAogICAgIiAgICBkaW5vX2NhY2hlcyA9IHByZWNvbXB1dGVfZGlub19jYWNoZShpbWFnZV9maWxlbmFtZXMsIGRpbm9fY2FjaGVfZGlyLCBleHRyYWN0b3IpXG5cbiIKICAgICIgICAgY29udGV4dF9kaW0gPSBleHRyYWN0b3IuZW1iZWRfZGltXG4iLAogICAgIiAgICBkaW5vX2NhY2hlcyA9IHByZWNvbXB1dGVfZGlub19jYWNoZShpbWFnZV9maWxlbmFtZXMsIGRpbm9fY2FjaGVfZGlyLCBleHRyYWN0b3IpXG4iCiAgICAiICAgIGNvbnRleHRfZGltID0gZXh0cmFjdG9yLmVtYmVkX2RpbVxuXG4iCiAgICAiICAgIGRlbCBleHRyYWN0b3JcbiIKICAgICIgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpXG4iCiAgICAiICAgIG5lcmZfbW9kZWwgPSBuZXJmX21vZGVsLnRvKGRldmljZSlcbiIsCiAgICAxLAopCnAud3JpdGVfdGV4dChzKQphc3QucGFyc2UocykKcHJpbnQoInRyYWluX2NvbmRpdGlvbmFsX25mLnB5OiBwYXRjaC1yZXMgY2FjaGUgKyBOZVJGLXBhcmsgKyBmcmFtZS1tYXRjaCIpCg=='
with open('/kaggle/temp/patch_dino_oom.py', 'wb') as fh:
    fh.write(base64.b64decode(_PATCH_B64))
with open('/kaggle/temp/setup.sh', 'w') as fh:
    fh.write(setup)
# pipefail so the cell sees bash's exit code, not tee's
rc = subprocess.call(
    ['bash', '-c', 'set -o pipefail; bash /kaggle/temp/setup.sh 2>&1 | tee /kaggle/working/setup.log'])
if rc != 0:
    print('--- last 60 lines of setup.log ---')
    print(subprocess.run(['tail', '-n', '60', '/kaggle/working/setup.log'], capture_output=True, text=True).stdout)
    raise RuntimeError(f'setup.sh failed (exit {rc}). See /kaggle/working/setup.log '
                       'and, for tiny-cuda-nn, /kaggle/working/tcnn_build.log')

In [ ]:
# @title 2. Config
import os, glob
TMP = '/kaggle/temp'
REPO = f'{TMP}/VF-NeRF-conditional'
VENV = f'{TMP}/venv310/bin'
WORK = '/kaggle/working'
SCENE = 'bonsai'
DATA_DIR = f'{REPO}/data/mipnerf360/{SCENE}'
NERF_OUTPUT_DIR = f'{WORK}/outputs'          # persisted
COND_NF_CKPT_DIR = f'{WORK}/checkpoints/conditional_nf/{SCENE}'  # persisted
RENDER_DIR = f'{WORK}/renders'               # persisted

# nerfacto recipe mirrored from the original VF-NeRF (leosegre/VF_NeRF,
# reg_pipeline_pc.py): downscale 2, 60k iters, 1024 rays/batch, camera-opt off,
# full train split, center-method=focus, scene-scale 2. Tweak here if needed.
NERFACTO_DOWNSCALE   = 2
NERFACTO_ITERS       = 60000
NERFACTO_RAYS        = 1024
NERFACTO_FORCE_RETRAIN = False   # True = ignore/delete any existing nerfacto run and retrain
COND_NF_MAX_STEPS    = 20000
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.7/lib64:' + os.environ.get('LD_LIBRARY_PATH','')
os.makedirs(RENDER_DIR, exist_ok=True)

# ---- optional: load checkpoints from an attached Kaggle Dataset ----
# Attach a Dataset (right sidebar -> Input -> Add Input) and drop files in it:
#   * a nerfacto checkpoint  -> skips cell 3 training. Either a *.tar.gz of an
#     outputs/ tree, OR (Kaggle auto-extracts archives on upload) the loose
#     nerfacto/<timestamp>/ run dir itself (config.yml + nerfstudio_models/*.ckpt).
#   * a conditional-NF checkpoint (latest.pt or cond_nf_step_*.pt) -> skips cell 5.
# Everything under /kaggle/input is auto-detected here.
PRETRAINED_TARS = sorted(glob.glob('/kaggle/input/**/*.tar.gz', recursive=True) +
                         glob.glob('/kaggle/input/**/*.tgz', recursive=True))
PRETRAINED_NERF_CFGS = sorted(
    c for c in glob.glob('/kaggle/input/**/config.yml', recursive=True)
    if os.sep + 'nerfacto' + os.sep in c
    and glob.glob(os.path.join(os.path.dirname(c), 'nerfstudio_models', '*.ckpt')))
INPUT_COND_PT   = sorted(glob.glob('/kaggle/input/**/latest.pt', recursive=True) +
                         glob.glob('/kaggle/input/**/cond_nf_step_*.pt', recursive=True))
print('nerfacto tarballs found :', PRETRAINED_TARS)
print('nerfacto run dirs found :', PRETRAINED_NERF_CFGS)
print('conditional-NF .pt found:', INPUT_COND_PT)

In [ ]:
# @title 3. Train (or restore) the frozen nerfacto backbone
import glob, subprocess, os, tarfile, re, shutil

def newest_config():
    for pat in (f'{NERF_OUTPUT_DIR}/{SCENE}/nerfacto/*/config.yml',
                f'{WORK}/**/nerfacto/*/config.yml'):
        c = sorted(glob.glob(pat, recursive=True))
        if c:
            return c[-1]
    return None

def repoint_output_dir(cfg_path, target=f'{WORK}/outputs'):
    # configs from another machine bake in an absolute output_dir; repoint it so
    # nerfstudio resolves <output_dir>/<exp>/nerfacto/<ts>/nerfstudio_models here.
    # nerfstudio serialises a Path as a multi-line !!python/object/apply:pathlib
    # .PosixPath + block-sequence-of-parts; a from-scratch config uses a plain
    # scalar. Pick the branch by which FORM is present -- not by whether re.sub
    # changed anything (a config already pointing here yields an identical sub).
    s = open(cfg_path).read()
    pathlib_pat = r'output_dir:(?: &\S+)? !!python/object/apply:pathlib\.PosixPath\n(?:- .*\n)+'
    block = ('output_dir: !!python/object/apply:pathlib.PosixPath\n'
             + ''.join(f'- {p}\n' for p in ['/'] + target.strip('/').split('/')))
    if re.search(pathlib_pat, s):
        s = re.sub(pathlib_pat, block, s, count=1)
    elif re.search(r'^output_dir: .+$', s, flags=re.M):
        s = re.sub(r'^output_dir: .+$', f'output_dir: {target}', s, count=1, flags=re.M)
    else:
        raise RuntimeError(f'no output_dir key found in {cfg_path}')
    open(cfg_path, 'w').write(s)

if NERFACTO_FORCE_RETRAIN and not (PRETRAINED_TARS or PRETRAINED_NERF_CFGS):
    shutil.rmtree(f'{NERF_OUTPUT_DIR}/{SCENE}', ignore_errors=True)
    print('NERFACTO_FORCE_RETRAIN: cleared', f'{NERF_OUTPUT_DIR}/{SCENE}')

NERF_CONFIG = newest_config()

if NERF_CONFIG is None and PRETRAINED_TARS:
    print('Restoring nerfacto from tarball', PRETRAINED_TARS[0])
    with tarfile.open(PRETRAINED_TARS[0]) as t:
        t.extractall(WORK)                       # -> {WORK}/outputs/{SCENE}/nerfacto/<ts>/
    for cfg in glob.glob(f'{WORK}/**/nerfacto/*/config.yml', recursive=True):
        repoint_output_dir(cfg)
    NERF_CONFIG = newest_config()

if NERF_CONFIG is None and PRETRAINED_NERF_CFGS:
    src_run = os.path.dirname(PRETRAINED_NERF_CFGS[-1])
    dst_run = f'{NERF_OUTPUT_DIR}/{SCENE}/nerfacto/{os.path.basename(src_run)}'
    print('Restoring nerfacto from loose run dir', src_run, '->', dst_run)
    os.makedirs(os.path.dirname(dst_run), exist_ok=True)
    shutil.copytree(src_run, dst_run, dirs_exist_ok=True)
    repoint_output_dir(f'{dst_run}/config.yml')
    NERF_CONFIG = newest_config()

NERF_RESTORED = NERF_CONFIG is not None   # restored/reused vs trained fresh below

if NERF_CONFIG:
    print('Using frozen nerfacto checkpoint:', NERF_CONFIG)
else:
    # nerfacto args mirrored from original VF-NeRF reg_pipeline_pc.py (minus the
    # stripped --nf-first-iter / --predict-view-likelihood which belonged to the
    # in-nerfacto NF that this fork replaced with a standalone trainer).
    cmd = [f'{VENV}/ns-train', 'nerfacto', '--data', DATA_DIR,
           '--output-dir', NERF_OUTPUT_DIR, '--vis', 'tensorboard',
           '--viewer.quit-on-train-completion', 'True',
           '--max-num-iterations', str(NERFACTO_ITERS),
           '--pipeline.datamanager.train-num-rays-per-batch', str(NERFACTO_RAYS),
           '--pipeline.datamanager.camera-optimizer.mode', 'off',
           'nerfstudio-data',
           '--downscale-factor', str(NERFACTO_DOWNSCALE),
           '--center-method', 'focus',
           '--orientation-method', 'up',
           '--auto-scale-poses', 'True',
           '--scene-scale', '2']
    # NOTE: original VF-NeRF also passed --train-split-fraction 1.0, but it had
    # explicit train/eval transform files. With a single transforms.json, 1.0
    # leaves 0 eval cameras -> nerfacto's periodic eval AND ns-render (cell 4)
    # both crash. Keeping the 0.9 default (~237 train imgs) instead.
    print(' '.join(cmd)); subprocess.run(cmd, check=True, cwd=REPO)
    NERF_CONFIG = newest_config()

assert NERF_CONFIG, 'no nerfacto config produced'
# sanity: the checkpoint file nerfstudio will look for must exist
_ck = glob.glob(os.path.join(os.path.dirname(NERF_CONFIG), 'nerfstudio_models', '*.ckpt'))
print('NERF_CONFIG =', NERF_CONFIG)
print('checkpoint  =', _ck[-1] if _ck else 'MISSING (restore path/layout wrong)')

In [ ]:
# @title 4. (optional) Render a spiral flythrough of the frozen NeRF
# Auto-skipped when nerfacto was restored from a checkpoint -- the flythrough is
# only useful to eyeball a fresh training run. Flip FORCE_RENDER to render anyway.
import subprocess, glob
FORCE_RENDER = False

if NERF_RESTORED and not FORCE_RENDER:
    print('nerfacto restored from a checkpoint -> skipping flythrough render.')
    print('set FORCE_RENDER = True in this cell to render it anyway.')
else:
    out = f'{RENDER_DIR}/{SCENE}_spiral.mp4'
    cmd = [f'{VENV}/ns-render', '--load-config', NERF_CONFIG, '--traj', 'spiral',
           '--rendered-output-names', 'rgb', 'depth', '--output-path', out, '--seconds', '6']
    print(' '.join(cmd))
    r = subprocess.run(cmd, cwd=REPO)
    print('spiral render exit', r.returncode)

print('renders present:', glob.glob(f'{RENDER_DIR}/*.mp4'))

In [ ]:
# @title 5. Train (or load) the conditional Normalizing Flow  ->  /kaggle/working/checkpoints/
import subprocess, os, shutil
os.makedirs(COND_NF_CKPT_DIR, exist_ok=True)
COND_LATEST = f'{COND_NF_CKPT_DIR}/latest.pt'

if INPUT_COND_PT:
    # train_conditional_nf.py has no resume flag, so a provided .pt is used as-is
    # (for the explorer / packaging); it is not continued.
    src = INPUT_COND_PT[-1]
    print('Using provided conditional-NF checkpoint:', src)
    if os.path.abspath(src) != os.path.abspath(COND_LATEST):
        shutil.copy(src, COND_LATEST)
else:
    cmd = [f'{VENV}/python', '-u', 'scripts/train_conditional_nf.py',
           '--nerf-config', NERF_CONFIG,
           '--scene-dir', DATA_DIR,
           '--checkpoint-dir', COND_NF_CKPT_DIR,
           '--max-steps', str(COND_NF_MAX_STEPS),
           '--batch-size', '4096']
    print(' '.join(cmd))
    # -u + PYTHONUNBUFFERED so the 'step N/20000 | loss ...' lines stream live
    # rather than sitting in a block buffer for minutes. The first ~5-10 min is
    # a silent DINO patch-feature precompute over every training image (a
    # per-image progress line prints as it goes).
    # NB: the NF loss is a NEGATIVE log-likelihood of a CONTINUOUS density -- it
    # legitimately goes negative (e.g. ~ -4) as the flow sharpens. Watch that
    # the trend is downward and finite, not the sign.
    subprocess.run(cmd, check=True, cwd=REPO,
                   env={**os.environ, 'PYTHONUNBUFFERED': '1'})
!ls -la {COND_NF_CKPT_DIR}

In [ ]:
# @title 6a. Pick up to 5 points (click on the image)
# Click directly on the displayed image to add a numbered red circle; each click
# appends [image_index, x, y] to /kaggle/working/coords.json (read by cell 6b).
# Pick from different images by changing BROWSE_INDEX and re-running (RESET=False
# keeps the points you already have). Set RESET=True to start over.
# Runs in the base kernel (no venv) via ipympl / %matplotlib widget. In a
# committed 'Save & Run All' run there is nobody to click -- it just writes
# DEFAULT_COORDS below, which cell 6b then uses.
import importlib.util, subprocess, sys, os, glob, json
if importlib.util.find_spec('ipympl') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipympl'], check=True)
    print('installed ipympl -- if the plot below is not interactive, re-run this cell once')
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception as e:
    print('could not enable the widget backend (%s); clicks disabled, using DEFAULT_COORDS' % e)
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from PIL import Image

BROWSE_INDEX = 0        # which images_2/* image to pick on right now
MAX_POINTS   = 5
RESET        = False    # True = discard picked points and start from DEFAULT_COORDS
DEFAULT_COORDS = [
    [0, 780, 520],
    [0, 300, 380],
]
COORDS_FILE = f'{WORK}/coords.json'

if RESET or not os.path.exists(COORDS_FILE):
    COORDS = [list(c) for c in DEFAULT_COORDS]
    json.dump(COORDS, open(COORDS_FILE, 'w'))
else:
    COORDS = json.load(open(COORDS_FILE))

_imgs = sorted(glob.glob(f'{DATA_DIR}/images_2/*'))
_im = Image.open(_imgs[BROWSE_INDEX])
print(f'[{BROWSE_INDEX}] {_imgs[BROWSE_INDEX].split(chr(47))[-1]}   (W,H)={_im.size}   {len(COORDS)}/{MAX_POINTS} points')

_fig, _ax = plt.subplots(figsize=(11, 8))
_ax.imshow(_im)
_ax.set_title('click to add a point (max 5) - set RESET=True and re-run to clear')

def _redraw():
    for _p in list(_ax.patches): _p.remove()
    for _t in list(_ax.texts): _t.remove()
    _r = max(_im.size) // 45
    for _k, (_i, _x, _y) in enumerate(COORDS):
        if _i != BROWSE_INDEX:
            continue
        _ax.add_patch(Circle((_x, _y), radius=_r, fill=False, color='red', lw=2))
        _ax.text(_x + _r, _y - _r, str(_k), color='red', fontsize=13, weight='bold')
    _fig.canvas.draw_idle()

def _on_click(event):
    if event.inaxes is not _ax or event.xdata is None:
        return
    if len(COORDS) >= MAX_POINTS:
        print(f'already at {MAX_POINTS} points; set RESET=True and re-run to start over')
        return
    _x, _y = int(round(event.xdata)), int(round(event.ydata))
    COORDS.append([BROWSE_INDEX, _x, _y])
    json.dump(COORDS, open(COORDS_FILE, 'w'))
    print(f'point {len(COORDS) - 1}: img {BROWSE_INDEX} @ ({_x}, {_y})   {len(COORDS)}/{MAX_POINTS}')
    _redraw()

_fig.canvas.mpl_connect('button_press_event', _on_click)
_redraw()

In [ ]:
EXPLORER_SRC = r'''"""Headless point-and-generate explorer for the conditional-NF fork.

This is the committed-run ("Save & Run All") path for probing the conditional NF:
it reproduces app/gradio_app.py's workflow without a UI. For each (image_index,
x, y) probe, take that pixel's DINOv2 patch-token feature as the NF condition,
sample candidate (point, direction) pairs, rank by likelihood, render the top
few through the frozen NeRF, and write a montage PNG. (Points come from the
notebook's cell 6a; run app/gradio_app.py directly on a local GPU box for a live
click UI.)

Runs inside the py3.10 venv (needs torch 1.13 + nerfstudio + tinycudann); the
notebook cell that launches it displays the PNGs from the base kernel.
"""
import argparse
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import torch

REPO = "/kaggle/temp/VF-NeRF-conditional"
sys.path.insert(0, REPO)

from nerfstudio.cameras.cameras import Cameras
from nerfstudio.fields.nf_field import ConditionalNFField
from nerfstudio.utils.dino_features import DinoExtractor, load_image_chw_01
from nerfstudio.utils.eval_utils import eval_setup

# nerfstudio's default post-auto-orient world-up axis (orientation-method "up").
DEFAULT_WORLD_UP = torch.tensor([0.0, 0.0, 1.0])


# --- helpers copied from app/gradio_app.py (kept gradio-free) ------------------

def default_backoff_distance(cameras: Cameras) -> float:
    centers = cameras.camera_to_worlds[..., :3, 3]
    scene_center = centers.mean(dim=0)
    return (centers - scene_center).norm(dim=-1).median().item()


def build_camera_from_point_direction(position, direction, reference_cameras, backoff_distance,
                                      world_up=DEFAULT_WORLD_UP):
    device = position.device
    world_up = world_up.to(device)
    forward = direction / direction.norm().clamp_min(1e-8)
    up_ref = world_up
    if torch.abs(torch.dot(forward, up_ref)) > 0.99:
        up_ref = torch.tensor([1.0, 0.0, 0.0], device=device)
    right = torch.cross(forward, up_ref)
    right = right / right.norm().clamp_min(1e-8)
    up = torch.cross(right, forward)
    camera_origin = position - forward * backoff_distance
    rotation = torch.stack([right, up, -forward], dim=-1)
    c2w = torch.cat([rotation, camera_origin.unsqueeze(-1)], dim=-1)
    return Cameras(
        camera_to_worlds=c2w.unsqueeze(0),
        fx=reference_cameras.fx[0:1], fy=reference_cameras.fy[0:1],
        cx=reference_cameras.cx[0:1], cy=reference_cameras.cy[0:1],
        width=reference_cameras.width[0:1], height=reference_cameras.height[0:1],
        camera_type=reference_cameras.camera_type[0:1],
    ).to(device)


def load_conditional_nf(checkpoint_path: Path, device: torch.device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    field = ConditionalNFField(
        context_dim=ckpt["context_dim"],
        num_dims=ckpt.get("num_dims", 6),
        num_blocks=ckpt["num_blocks"],
        hidden_dim=ckpt["hidden_dim"],
        cond_prior=ckpt["cond_prior"],
        use_cond_in_coupling=True,
        use_batchnorm=ckpt["use_batchnorm"],
        device=str(device),
    )
    field.load_state_dict(ckpt["model_state"])
    field.eval()
    return field, ckpt["dino_model_name"], int(ckpt.get("step", -1))


# --- main --------------------------------------------------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--nerf-config", required=True)
    ap.add_argument("--scene-dir", required=True)
    ap.add_argument("--cond-nf-checkpoint", required=True)
    ap.add_argument("--probes", required=True, help='JSON list of [image_index, x, y]')
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--num-samples", type=int, default=200)
    ap.add_argument("--render-top", type=int, default=5)
    ap.add_argument("--render-downscale", type=float, default=3.0)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Loading frozen NeRF from {args.nerf_config} ...", flush=True)
    config, pipeline, _, _ = eval_setup(Path(args.nerf_config), test_mode="inference")
    nerf_model = pipeline.model.to(device).eval()
    for p in nerf_model.parameters():
        p.requires_grad_(False)

    # Build the reference cameras in the SAME frame the frozen NeRF (and hence the
    # conditional NF) was trained in -- the checkpoint's own dataparser config,
    # not stock defaults (center-method / scene-scale differ).
    dp = config.pipeline.datamanager.dataparser
    dp.data = Path(args.scene_dir)
    outs = dp.setup().get_dataparser_outputs(split="train")
    cameras = outs.cameras.to(device)
    image_filenames = [Path(p) for p in outs.image_filenames]
    backoff = default_backoff_distance(cameras)
    print(f"{len(image_filenames)} training images | backoff {backoff:.3f}", flush=True)

    field, dino_model_name, step = load_conditional_nf(Path(args.cond_nf_checkpoint), device)
    print(f"conditional-NF checkpoint step = {step}", flush=True)
    extractor = DinoExtractor(model_name=dino_model_name, device=str(device))

    grid_cache = {}

    def patch_feature(i, x, y):
        """Pixel (x, y) on the dataparser image i -> its DINOv2 patch-token feature.

        Indexes the patch grid directly (same as the NF trainer's sample_batch);
        no giant per-pixel upsampled map.
        """
        if i not in grid_cache:
            img = load_image_chw_01(image_filenames[i])
            with torch.no_grad():
                g, (h, w) = extractor.extract_patch_grid(img)
            grid_cache[i] = (g.float(), img, h, w)
        g, img, h, w = grid_cache[i]
        hp, wp = g.shape[-2:]
        py = min(max(int(y * hp / h), 0), hp - 1)
        px = min(max(int(x * wp / w), 0), wp - 1)
        return g[:, py, px].reshape(-1).to(device), img

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    probes = json.loads(args.probes)
    manifest = []

    for k, (i, x, y) in enumerate(probes):
        i, x, y = int(i), int(x), int(y)
        cond, img = patch_feature(i, x, y)
        with torch.no_grad():
            samples = field.sample(num_samples=args.num_samples, context=cond)
            logp = field.log_prob(
                samples, cond.unsqueeze(0).expand(args.num_samples, -1)
            ).squeeze(-1)
        order = torch.argsort(logp, descending=True)[: args.render_top]
        n = len(order)

        fig, ax = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
        ax = [ax] if n == 0 else list(ax)
        ax[0].imshow(img.permute(1, 2, 0).numpy())
        ax[0].add_patch(Circle((x, y), radius=max(img.shape[-2:]) / 45,
                               fill=False, color="red", lw=2.5))
        ax[0].set_title(f"img {i} @ ({x},{y})")
        ax[0].axis("off")
        for j, idx in enumerate(order):
            cam = build_camera_from_point_direction(
                samples[idx, :3], samples[idx, 3:], cameras, backoff
            )
            cam.rescale_output_resolution(1.0 / args.render_downscale)
            with torch.no_grad():
                o = nerf_model.get_outputs_for_camera_ray_bundle(
                    cam.generate_rays(camera_indices=0)
                )
            ax[j + 1].imshow(o["rgb"].clamp(0, 1).cpu().numpy())
            ax[j + 1].set_title(f"logp {logp[idx].item():.2f}")
            ax[j + 1].axis("off")

        png = out_dir / f"probe_{k:02d}.png"
        fig.tight_layout()
        fig.savefig(png, dpi=90)
        plt.close(fig)
        manifest.append(str(png))
        print(f"probe {k}: img {i} ({x},{y}) [{image_filenames[i].name}] -> {png}", flush=True)

    print("MANIFEST " + json.dumps(manifest), flush=True)


if __name__ == "__main__":
    main()
'''

In [ ]:
# @title 6b. Generate novel views for the picked points
# For each point picked in cell 6a: take that pixel's DINOv2 patch-token feature
# as the NF condition, sample candidate (point, direction) pairs, rank them by
# likelihood, render the top few through the frozen NeRF, and show a montage
# (circled source point + rendered novel views) inline. Re-run after editing 6a.
# Works in a committed 'Save & Run All' run too (6a seeds coords.json from
# DEFAULT_COORDS); writes PNGs, never blocks.
import subprocess, os, json, glob
from IPython.display import Image as _Img, display

COORDS = json.load(open(f'{WORK}/coords.json'))
assert COORDS, 'no points -- run cell 6a first'
print('points:', COORDS)
NUM_SAMPLES, RENDER_TOP, RENDER_DOWNSCALE = 200, 5, 3.0

EXPLORE_DIR = f'{WORK}/explorer'
os.makedirs(EXPLORE_DIR, exist_ok=True)
for _p in glob.glob(f'{EXPLORE_DIR}/probe_*.png'):
    os.remove(_p)
subprocess.run([f'{VENV}/pip', 'install', '-q', 'matplotlib'], check=True)
with open('/kaggle/temp/explorer.py', 'w') as fh:
    fh.write(EXPLORER_SRC)
cmd = [f'{VENV}/python', '-u', '/kaggle/temp/explorer.py',
       '--nerf-config', NERF_CONFIG, '--scene-dir', DATA_DIR,
       '--cond-nf-checkpoint', f'{COND_NF_CKPT_DIR}/latest.pt',
       '--probes', json.dumps(COORDS), '--out-dir', EXPLORE_DIR,
       '--num-samples', str(NUM_SAMPLES), '--render-top', str(RENDER_TOP),
       '--render-downscale', str(RENDER_DOWNSCALE)]
subprocess.run(cmd, check=True, cwd=REPO,
               env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for png in sorted(glob.glob(f'{EXPLORE_DIR}/probe_*.png')):
    print(png)
    display(_Img(filename=png))

In [ ]:
# @title 7. Package outputs
import shutil, os, glob
# copy the (small) nerfstudio_models + config out of the run dir into a flat tree,
# then zip everything persisted.
bundle = f'{WORK}/vf_nerf_outputs'
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle, exist_ok=True)
run_dir = os.path.dirname(NERF_CONFIG)
shutil.copytree(run_dir, f'{bundle}/nerfacto_{SCENE}',
                ignore=shutil.ignore_patterns('*.tfevents*'))
shutil.copytree(COND_NF_CKPT_DIR, f'{bundle}/conditional_nf_{SCENE}')
for mp4 in glob.glob(f'{RENDER_DIR}/*.mp4'): shutil.copy(mp4, bundle)
for png in glob.glob(f'{WORK}/explorer/probe_*.png'): shutil.copy(png, bundle)
shutil.make_archive(bundle, 'zip', bundle)
print('bundle:', bundle + '.zip')
!du -sh {bundle}.zip {WORK}/renders {WORK}/checkpoints {WORK}/outputs
!find {bundle} -maxdepth 3 -type f | sort

## Getting your results

**Committed run (Save & Run All):** when the version finishes, open it and use the
**Output** tab — download `vf_nerf_outputs.zip` (nerfacto checkpoint + config +
conditional-NF `latest.pt` + the flythrough videos), or the individual files.

**Interactive run:** the file browser on the right shows `/kaggle/working/` — right-click
→ Download. Do a **Save Version** first so the outputs are also stored server-side.

If the GUI file browser / Output tab is unavailable, get a direct download link from a cell:

```python
from IPython.display import FileLink
import shutil; shutil.make_archive('/kaggle/working/vf_nerf_outputs', 'zip', '/kaggle/working/vf_nerf_outputs')
FileLink('vf_nerf_outputs.zip')   # click it (works while the session is alive)
```

## Loading an existing checkpoint into this notebook

Kaggle notebooks can only read outside files through an attached **Dataset** (or Secret).
There is no Drive mount. So:

1. **kaggle.com -> Create -> New Dataset.** Upload your checkpoint file(s):
   - a **nerfacto** backup: either a `*.tar.gz` of an `outputs/` tree (what the earlier
     Colab `tar_ckpts.py` produced), **or** just the loose `nerfacto/<timestamp>/` run
     dir (`config.yml` + `nerfstudio_models/*.ckpt` + `dataparser_transforms.json`).
     Either works -- Kaggle auto-extracts an uploaded archive, so a `.tar.gz` ends up
     as the loose tree anyway. **or**
   - a **conditional-NF** checkpoint: `latest.pt` or `cond_nf_step_*.pt` from
     `scripts/train_conditional_nf.py`.
   Set it Private if you like; give it any title.
2. In this notebook: right sidebar -> **Input -> Add Input** -> your dataset -> **Add**.
   It mounts read-only at `/kaggle/input/<dataset-slug>/`.
3. Re-run from **cell 2**. It auto-detects the files:
   - nerfacto tarball or loose run dir found -> **cell 3 copies/extracts it, repoints the
     baked-in `output_dir`, and skips nerfacto training**;
   - `.pt` found -> **cell 5 copies it to `latest.pt` and skips NF training** (the trainer
     has no resume flag, so a `.pt` is only *used*, not continued).

The `bonsai` scene still downloads in cell 1 either way (needed for DINO features, the
render camera path, and the dataparser the config points at).

To get files *out* to Google Drive there's no built-in mount — download
`vf_nerf_outputs.zip` and upload it yourself, or add an `rclone`/Drive-API cell backed by
a Kaggle Secret.

## Interactive viewing — what works where

| tool | what it shows | Kaggle | needs |
|---|---|---|---|
| cell 4 video | spiral flythrough of the reconstructed NeRF | ✅ committed or interactive — **optional**, auto-skipped when nerfacto is restored (set `FORCE_RENDER=True`) | nothing extra |
| cells 6a/6b | click points on a training image (6a) → NF samples novel-view rays → render top few (montage PNGs, 6b) | ✅ committed **or** interactive (6a clicks need an interactive session; committed runs use `DEFAULT_COORDS`) | both checkpoints |
| `ns-viewer` | free 3D navigation of the NeRF | ❌ not from Kaggle | browser must reach `ws://localhost:<port>` — i.e. local machine, or SSH port-forward from a rented GPU box (RunPod/Vast/Lambda) |

So: for a quick look at the NeRF itself, force the cell 4 spiral render. To probe the
conditional NF, click up to 5 points on an image in **cell 6a** (or edit `DEFAULT_COORDS`
there for a committed run), then run **cell 6b** to generate the sampled novel views for
them. A live click UI (`app/gradio_app.py`) is available if you run this repo on a local
GPU box; on Kaggle use cells 6a/6b. Full free-fly `ns-viewer` needs a machine you can
point a browser at `localhost` on (local, or `ssh -L 7007:localhost:7007` into a rented
GPU box) — not Kaggle or Colab.